# 85 · Dashboard evidence verification

Offline and reproducible. Re-derives the dashboard's numbers from the original archives (when present) instead of trusting the index. No GPU, no network. Set `CELL_AKAMAI_ARCHIVES` to the folder holding the returned zips (default `~/Downloads`). Absent archives are reported as `NOT_RECOVERED`, never as PASS.

In [ ]:
import glob, hashlib, io, json, os, re, zipfile
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd()/"evidence").exists() else Path.cwd().parent
ARCH = Path(os.environ.get("CELL_AKAMAI_ARCHIVES", Path.home()/"Downloads"))
vs = sorted(glob.glob(str(ROOT/"evidence/dashboard/DASHBOARD_EVIDENCE_INDEX.v*.json")), key=lambda p: int(p.split(".v")[-1][:-5]))
IDX = json.loads(Path(vs[-1]).read_text()); ROWS = IDX["rows"]
print("INDEX", Path(vs[-1]).name, "rows", len(ROWS), "schema", IDX["schema"])

## 1. Archive identity

In [ ]:
checks = []
for s in IDX["sources"]:
    if s.get("state") == "NOT_RECOVERED":
        print(f"{s['label']:<22} NOT_RECOVERED ({s['pattern']})"); continue
    p = ARCH / s["archive"]
    if not p.exists():
        print(f"{s['label']:<22} ARCHIVE_NOT_PRESENT_HERE (cannot re-verify)"); continue
    ok = hashlib.sha256(p.read_bytes()).hexdigest() == s["sha256"]; checks.append(ok)
    print(f"{s['label']:<22} sha256 {'MATCH' if ok else 'MISMATCH'}  manifest {s.get('manifest_state')}")

## 2. Independent recomputation of Akamai server-side token deltas

In [ ]:
def gen_tokens(text):
    return sum(float(m) for m in re.findall(r"^vllm:generation_tokens_total\{[^}]*\}\s+([0-9.eE+-]+)$", text, re.M))
exp = ARCH / "CELL_AKAMAI_EVIDENCE_EXPORT.zip"
if not exp.exists():
    print("NOT_RECOVERED here: CELL_AKAMAI_EVIDENCE_EXPORT.zip")
else:
    z = zipfile.ZipFile(exp); n = 0
    for r in ROWS:
        if r["metric"] != "server_generation_tokens": continue
        bpath, apath = r["evidence_ref"].split(":", 1)[1].split("|")
        mine = gen_tokens(z.read(apath).decode()) - gen_tokens(z.read(bpath).decode())
        ok = abs(mine - r["value"]) < 1e-9; checks.append(ok); n += 1
        print(f"{r['experiment_id']} request {r['request']}: index {r['value']:.0f}  recomputed {mine:.0f}  {'MATCH' if ok else 'MISMATCH'}")
    print("recomputed", n, "deltas")

## 3. Six-dataset file hashes and reconstruction

In [ ]:
six = ARCH / "CELL_AKAMAI_M08_M09_SIX_DATASETS_20261012.zip"
if not six.exists():
    print("NOT_RECOVERED here: six-dataset archive")
else:
    z = zipfile.ZipFile(six); ver = json.loads(z.read("data/SIX_DATASET_VERIFICATION.json"))
    for role in ("genesis", "vita_daisy", "evaluator"):
        orig = [json.loads(l) for l in z.read(f"data/original/{role}.jsonl").decode().splitlines() if l.strip()]
        fco = [json.loads(l) for l in z.read(f"data/fco/{role}.jsonl").decode().splitlines() if l.strip()]
        recon = [f["payload"] for f in fco] == orig
        hashes = all(hashlib.sha256(z.read(f"data/{k}/{role}.jsonl")).hexdigest() == ver["files"][f"data/{k}/{role}.jsonl"] for k in ("original","fco","fcg"))
        checks += [recon, hashes]
        print(f"{role:<11} records {len(orig)}  reconstruction {'EQUAL' if recon else 'DIFFERENT'}  hashes {'MATCH' if hashes else 'MISMATCH'}  class ECA_DERIVED_FIXTURE")

## 4. Coverage and claim boundaries

In [ ]:
from collections import Counter
for (exp_id, state), n in sorted(Counter((r["experiment_id"], r["state"]) for r in ROWS).items()):
    print(f"{exp_id:<42} {state:<36} {n}")
for k, v in IDX["claims"].items(): print(f"CLAIM {k}: {v}")
print("RECOMPUTATION", "PASS" if checks and all(checks) else ("NOT_TESTED (no archives here)" if not checks else "FAIL"), f"({sum(checks)}/{len(checks)} checks)")